# 01 - Exploring the Flood Dataset

Walkthrough of the README, run against **all** scenes instead of just `image_0`.

Put this notebook in the repo root, next to the `data/` folder. Run cells top to bottom.

**Goals for this notebook**
1. Confirm image, DEM, and label line up (same size, CRS, grid)
2. Look at every scene
3. Measure flood vs. dry balance
4. Check DEM values
5. Figure out what the nodata value in `label.tif` actually is

## 0. Setup

In [ ]:
# Run once in the VS Code terminal (with .venv active) if you haven't:
# pip install rasterio numpy matplotlib pandas

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

DATA_DIR = Path("data")
OUT_DIR = Path("outputs/exploration")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Find every scene folder (image_0, image_1, ...) and sort them by number
scenes = sorted(
    [p for p in DATA_DIR.glob("image_*") if p.is_dir()],
    key=lambda p: int(p.name.split("_")[-1]),
)
print(f"Found {len(scenes)} scenes:", [s.name for s in scenes])

## 1. Metadata and alignment check

README step 1, but for every scene. The `aligned` column should be `True` everywhere.
If it's `False`, the image/DEM/label grids don't match and you can't just stack them as channels.

In [ ]:
rows = []
for scene in scenes:
    with rasterio.open(scene / "image.tif") as img, \
         rasterio.open(scene / "dem.tif") as dem, \
         rasterio.open(scene / "label.tif") as lab:
        aligned = (
            img.width == dem.width == lab.width
            and img.height == dem.height == lab.height
            and img.crs == dem.crs == lab.crs
            and img.transform.almost_equals(dem.transform)
            and img.transform.almost_equals(lab.transform)
        )
        rows.append({
            "scene": scene.name,
            "bands": img.count,
            "width": img.width,
            "height": img.height,
            "img_dtype": img.dtypes[0],
            "pixel_size": tuple(round(r, 3) for r in img.res),
            "crs": str(img.crs),
            "img_nodata": img.nodata,
            "dem_nodata": dem.nodata,
            "label_dtype": lab.dtypes[0],
            "label_nodata": lab.nodata,
            "aligned": aligned,
        })

meta = pd.DataFrame(rows)
meta.to_csv(OUT_DIR / "metadata.csv", index=False)
meta

In [ ]:
bad = meta[~meta["aligned"]]
print("All scenes aligned!" if bad.empty else f"Misaligned scenes:\n{bad[['scene','width','height','crs']]}")

## 2. Look at it

README step 2. One figure per scene: RGB, DEM, label side by side, saved to `outputs/exploration/`.

Things to eyeball:
- Does the blue in the label match what looks like water in the RGB?
- Do flooded areas sit in low spots on the DEM?

In [ ]:
def stretch(band, low=2, high=98):
    """Percentile stretch so imagery isn't too dark (works for uint8 or uint16)."""
    lo, hi = np.nanpercentile(band, (low, high))
    return np.clip((band - lo) / max(hi - lo, 1e-6), 0, 1)

def label_nodata_mask(label, nodata):
    """True where the label is NOT dry (0) or flood (1). Catches -1, 255, or whatever is used."""
    mask = ~np.isin(label, [0, 1])
    if nodata is not None:
        mask |= (label == nodata)
    return mask

def plot_scene(scene, save=True, show=True):
    with rasterio.open(scene / "image.tif") as src:
        image = src.read().astype("float32")  # (bands, H, W)
    with rasterio.open(scene / "dem.tif") as src:
        dem = src.read(1, masked=True)
    with rasterio.open(scene / "label.tif") as src:
        label = src.read(1)
        lab_nodata = src.nodata

    if image.shape[0] >= 3:
        rgb = np.dstack([stretch(image[i]) for i in range(3)])
    else:
        rgb = stretch(image[0])

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(rgb, cmap=None if rgb.ndim == 3 else "gray")
    axes[0].set_title(f"{scene.name}: image")
    im = axes[1].imshow(dem, cmap="terrain")
    axes[1].set_title("DEM (elevation)")
    fig.colorbar(im, ax=axes[1], fraction=0.046)
    axes[2].imshow(np.ma.masked_where(label_nodata_mask(label, lab_nodata), label),
                   cmap="Blues", vmin=0, vmax=1)
    axes[2].set_title("Label (dark = flood, gray = nodata)")
    axes[2].set_facecolor("lightgray")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    if save:
        fig.savefig(OUT_DIR / f"{scene.name}_preview.png", dpi=120)
    if show:
        plt.show()
    else:
        plt.close(fig)

# Start with one scene
plot_scene(scenes[0])

In [ ]:
# Then all of them (saved to outputs/exploration/)
for scene in scenes:
    plot_scene(scene)

## 3. Class balance (flood vs. dry)

README step 3. The README's version reads from `processed_data/manifest.csv`, which doesn't exist yet,
so this loops over the scene folders directly. It also gives a per-scene breakdown, which matters
for picking the held-out test region.

In [ ]:
rows = []
for scene in scenes:
    with rasterio.open(scene / "label.tif") as src:
        label = src.read(1)
        nodata_mask = label_nodata_mask(label, src.nodata)
    dry = int(((label == 0) & ~nodata_mask).sum())
    flood = int(((label == 1) & ~nodata_mask).sum())
    nodata = int(nodata_mask.sum())
    known = dry + flood
    rows.append({
        "scene": scene.name,
        "dry": dry,
        "flood": flood,
        "nodata": nodata,
        "flood_%_of_known": round(100 * flood / known, 2) if known else None,
        "nodata_%_of_total": round(100 * nodata / label.size, 2),
    })

balance = pd.DataFrame(rows)
total_known = balance["dry"].sum() + balance["flood"].sum()
print(f"Overall flood %: {100 * balance['flood'].sum() / total_known:.2f}% of known pixels")
balance.to_csv(OUT_DIR / "class_balance.csv", index=False)
balance

In [ ]:
ax = balance.plot.bar(x="scene", y="flood_%_of_known", legend=False, figsize=(10, 4))
ax.set_ylabel("% flooded (of known pixels)")
ax.set_title("Flood fraction per scene")
plt.tight_layout()
plt.savefig(OUT_DIR / "flood_fraction_per_scene.png", dpi=120)
plt.show()

## 4. DEM value range

README step 4. Elevation is usually in meters. Watch for crazy values like -9999 or -32768
that aren't flagged as nodata, those would wreck normalization later.

In [ ]:
rows = []
for scene in scenes:
    with rasterio.open(scene / "dem.tif") as src:
        dem = src.read(1, masked=True)  # respects nodata if it's set
    rows.append({
        "scene": scene.name,
        "min": float(dem.min()),
        "max": float(dem.max()),
        "mean": round(float(dem.mean()), 1),
        "std": round(float(dem.std()), 1),
        "masked_px": int(np.ma.count_masked(dem)),
    })

dem_stats = pd.DataFrame(rows)
dem_stats.to_csv(OUT_DIR / "dem_stats.csv", index=False)
dem_stats

## 5. What is the label's nodata value, really?

Heads up: the README isn't consistent. The folder tree and step 5 say nodata is **-1**,
but the plotting and class-balance code use **255**. Also, if `label.tif` is `uint8`,
it physically can't store -1. This cell shows what's actually in the files.

In [ ]:
all_values = {}
for scene in scenes:
    with rasterio.open(scene / "label.tif") as src:
        values, counts = np.unique(src.read(1), return_counts=True)
    for v, c in zip(values.tolist(), counts.tolist()):
        all_values[v] = all_values.get(v, 0) + c

print("Unique label values across all scenes (value: pixel count):")
for v, c in sorted(all_values.items()):
    print(f"  {v}: {c:,}")
print("\nlabel dtypes:", meta["label_dtype"].unique().tolist())
print("label nodata tags:", meta["label_nodata"].unique().tolist())

Whatever that value turns out to be, the plan for training is the same: remap it to one
`IGNORE` value and tell the loss to skip it. Quick demo of how that works:

In [ ]:
IGNORE = -1  # what we'll use in training code

def clean_label(label, nodata):
    out = label.astype("int64")  # int64 so -1 is allowed and torch is happy
    out[label_nodata_mask(label, nodata)] = IGNORE
    return out

with rasterio.open(scenes[0] / "label.tif") as src:
    cleaned = clean_label(src.read(1), src.nodata)
print("Values after cleaning:", np.unique(cleaned))

# In training:
# import torch.nn as nn
# criterion = nn.CrossEntropyLoss(ignore_index=IGNORE)
# and for metrics: valid = target != IGNORE, then compare pred[valid] vs target[valid]

## 6. Notes for the team

Fill this in after running everything, then paste it on the Trello card.

- **Alignment:** all scenes aligned? (yes / no, which ones)
- **Bands:** how many bands in image.tif? RGB order?
- **Label nodata value:** -1, 255, or something else?
- **Overall flood %:**
- **Scenes with very little flood:**
- **DEM weirdness:**
- **Candidate held-out test region:** (pick one with a decent amount of flood)
- **Imbalance plan:** (weighted loss / Dice loss / oversample flood tiles)